# Brain-1 - Kaggle training lane

Trains the 33.3M-parameter decoder-only model from `spec/model_config.toml` on a Kaggle accelerator, starting
from a container that resets between sessions.

**Order of operations, and it is not negotiable:** parity before speed, learning before scale. A green test
suite is not evidence the model can learn, and a fast kernel with wrong numbers is worthless. Every cell below
either proves something or says that it does not.

| # | cell | what it establishes |
|---|---|---|
| 1 | clone + `scripts/setup_colab.sh` | the pinned environment, rebuilt from scratch |
| 2 | `scripts/check_gpu.sh` | device, compute capability, fp16/bf16 support, hardware label |
| 3 | `pytest -q` | every backend matches the golden tensors before any timing is quoted |
| 4 | prepare + `brainlab leak` | the val split is really held out and the ratio hit its target |
| 5 | overfit one batch | the model can memorise 64 tokens, i.e. it *can* learn |
| 6 | short timed run | measured tokens/sec, used to project the full run against the session cap |
| 7 | the run, with budget and resume | the checkpoint cursor restores; losses do not replay |
| 8 | `brainlab ppl` + sampling | a comparable number, and the logits dtype behind it |

## Honest expectations

- No validated pretrained checkpoint is published in the repository. This notebook is how one gets made.
- The byte-level `--charlevel` tokenizer has a 258-token vocabulary while the model head is sized for 16,384.
  A perplexity from the charlevel route is **not** comparable to a trained-BPE number. Do not mix them.
- A Kaggle GPU session is capped at 9 h and can end sooner. Cell 6 measures throughput and projects whether
  8,000 steps fit; cell 7 stops itself before the cap and prints the resume line.
- Only `/kaggle/working` survives a session. `--out` must live there.
- Checkpoints are hundreds of MB. They belong in notebook output or a Kaggle Dataset, never in `git`.

## Notebook settings

Accelerator: **GPU T4 Premium** (or P100). Internet: **on**, for the clone and the install. Once cell 1 has
run, everything after it works with internet off, because the corpus route below reads attached Kaggle Datasets
rather than downloading from Hugging Face.

In [ ]:
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "Brain-1"
DATA = WORK / "data" / "packed"
CKPT = WORK / "ckpt"
print("repo:", REPO)
print("data:", DATA)
print("ckpt:", CKPT)

In [ ]:
import subprocess

def sh(args, cwd=None):
    proc = subprocess.run([str(item) for item in args], cwd=str(cwd), capture_output=True, text=True)
    return proc.returncode, proc.stdout.strip(), proc.stderr.strip()


SENTINEL = REPO / "python" / "brainlab" / "cli.py"
rc, out, err = 0, "", ""

if SENTINEL.exists():
    # Kaggle restores /kaggle/working from saved output, so a repo already on disk is usable even with
    # internet off. Try to fast-forward it, but never throw away code over a failed pull.
    rc, out, err = sh(["git", "-C", REPO, "pull", "--ff-only"], cwd=REPO)
    print("on disk from a previous session" + (": pull succeeded" if rc == 0 else f": pull failed ({rc}), using what is there"))
    if rc != 0:
        print(err)
    rc = 0
elif REPO.exists() and any(REPO.iterdir()):
    raise SystemExit(
        f"{REPO} exists but has no brainlab package in it - Kaggle restored output over an empty checkout. "
        "Move that directory aside (or start a fresh notebook) so the clone has somewhere to land."
    )
else:
    rc, out, err = sh(["git", "clone", "--depth", "1", "https://github.com/AHK811/Brain-1.git", REPO])

if rc != 0:
    print(out)
    print(err)
    raise SystemExit(
        f"git exited {rc}. If the message mentions name resolution or a timeout, Kaggle's internet is off: "
        "Settings > Internet > ON, then re-run this cell. Everything after the install runs with internet "
        "off, because the corpus route reads attached Datasets rather than downloading."
    )
print(sh(["git", "-C", REPO, "log", "--oneline", "-1"])[1])

In [ ]:
%cd /kaggle/working/Brain-1

## 1. Environment

`scripts/setup_colab.sh` is the one install script for every disposable container, so Kaggle uses it
unmodified. It reads the Rust pin out of `rust-toolchain.toml` instead of hard-coding a second, contradictory
version, and writes a hardware label to `bench/results/env.json` that the benchmark harness folds into every
result row.

Two switches matter. `BRAINLAB_MOJO_INSTALL=0` skips the Modular toolkit - the Mojo lane is deliberately
unmeasured and installing it costs minutes. Kaggle ships a CUDA-enabled PyTorch, so `BRAINLAB_FORCE_TORCH=1`
stays off on purpose: swapping a working GPU build for a CPU-pinned one is how a run quietly ends up on 40 CPU
cores. The script keeps the preinstalled torch and says so in its log.

In [ ]:
%%bash
set -uo pipefail
cd /kaggle/working/Brain-1
export BRAINLAB_MOJO_INSTALL=0
bash scripts/setup_colab.sh

`setup_colab.sh` creates its own venv, which did not exist when the cell above started, so resolve the
interpreter *after* the install. Every later cell shells out from Python with this path - a `%%bash` cell
cannot see a Python variable, and silently picking up the system interpreter instead of the venv is exactly the
kind of drift that makes a notebook non-reproducible.

In [ ]:
import json
import os
import subprocess
import sys

VENV_PY = REPO / ".venv" / "bin" / "python"
PY = VENV_PY if VENV_PY.exists() else Path(sys.executable)


def run(args, cwd=REPO, check=False):
    proc = subprocess.run([str(a) for a in args], cwd=str(cwd), capture_output=True, text=True)
    if check and proc.returncode != 0:
        print(proc.stdout[-3000:])
        print(proc.stderr[-3000:])
        raise SystemExit(f"{' '.join(str(a) for a in args)} exited {proc.returncode}")
    return proc


print("python:", PY)
print(run([PY, "-c", "import torch, numpy; print(torch.__version__, torch.version.cuda, numpy.__version__)"]).stdout.strip())
env_file = REPO / "bench" / "results" / "env.json"
if env_file.exists():
    print("pinned:", json.loads(env_file.read_text())["pinned"])
else:
    raise SystemExit("setup_colab.sh did not finish (no bench/results/env.json). Read "
                     "bench/results/setup_colab.log, fix the failing step, and re-run the install cell - "
                     "every later cell assumes the pinned environment it writes.")

## 2. Hardware label

`check_gpu.sh` records fp16 and bf16 capability instead of assuming either. A Tesla T4 is compute capability
7.5: **fp16 with GradScaler, no bf16.** That is why the recipe stays on fp16 and why nothing here reaches for
`torch.bfloat16`. The script exits 1 with no CUDA device, which this cell turns into a hard stop - continuing
would print CPU numbers under a GPU heading.

In [ ]:
proc = run(["bash", "scripts/check_gpu.sh"])
print(proc.stdout)
if proc.returncode != 0:
    raise SystemExit("no accelerator visible. Switch the notebook to GPU in Settings > Accelerator "
                     "and re-run; CPU throughput is not what this notebook is for.")

label = json.loads((REPO / "bench" / "results" / "gpu_check.json").read_text())
print(json.dumps(label, indent=2, sort_keys=True))

HAS_GPU = label["device_count"] > 0
DEVICE = "cuda" if HAS_GPU else "cpu"
DTYPE = "fp16" if HAS_GPU else "fp32"
if HAS_GPU and label["compute_capability"] == "7.5":
    print("\nT4 class: fp16 + GradScaler, no bf16, ~320 GB/s ceiling on memory-bound ops")
if HAS_GPU and not label["fp16"]:
    raise SystemExit("an accelerator is visible but an fp16 matmul failed; not starting a run that cannot train")
print(f"\ndevice={DEVICE} dtype={DTYPE} bf16={label['bf16']} cc={label['compute_capability']} "
      f"peak={label['peak_gbytes_per_s']} GB/s")

## 3. Parity before speed

Nothing past this line is worth timing until it passes. The golden tensors come from the PyTorch reference and
the harness - not the backend - owns the verdict, so a backend cannot grade its own homework. Skips are
expected and named: GPU-only rows on a CPU host, the Rust release binary when cargo did not build it, and the
rust-trained tokenizer cross-check without `BRAINLAB_TOKENIZER_DIR`.

In [ ]:
proc = run([PY, "-m", "pytest", "-q"])
print(proc.stdout[-4000:])
if proc.returncode != 0:
    raise SystemExit("parity is not green. Do not train on a tree whose backends disagree with the reference.")
print("parity green - proceed")

## 4. Data

Three routes, and only two work here:

- **Attached Kaggle Dataset** - works with internet off. `brainlab.data.prepare` walks a directory recursively
  for `.txt`, so attach a plain-text dataset and point `--corpus` at `/kaggle/input/<slug>`.
- **`--synthetic N`** - a deterministic corpus built in-process. Exercises the whole pipeline and proves the
  model learns; produces no usable language model.
- **Hugging Face download** - not a working path yet. `brainlab.data.fetch` only accepts `.txt` files in a repo,
  and `salesforce/wikitext` and `roneneldan/TinyStories` both ship as Parquet. Parquet ingestion is an open
  roadmap item. So when no `.txt` input is attached, the next cell says so and falls back to synthetic rather
  than pretending to train on real language.

`--doc-sep` is a choice, not an assumption: `blank` splits on blank lines, `line` on every newline, `file`
treats one file as one document. Getting this wrong merges unrelated documents into a single sequence, which
shows up as a model that predicts document transitions beautifully and nothing else.

In [ ]:
inputs = Path("/kaggle/input")
found = []
if inputs.exists():
    for slug in sorted(inputs.iterdir()):
        if not slug.is_dir():
            continue
        texts = [p for p in slug.rglob("*") if p.suffix.lower() == ".txt"]
        if texts:
            total = sum(p.stat().st_size for p in texts)
            found.append((slug, len(texts), total))
            print(f"{slug.name}: {len(texts)} .txt files, {total / 1e6:.1f} MB")

if found:
    slug, count, total = max(found, key=lambda item: item[2])
    CORPUS_ARGS = ["--corpus", str(slug)]
    SYNTHETIC = False
    print(f"\nusing {slug.name}: {count} files, {total / 1e6:.1f} MB")
else:
    # One synthetic document becomes one file and val is chosen by file-level stride, so the configured
    # 0.5% ratio leaves this route with too few validation sequences to evaluate anything at all.
    CORPUS_ARGS = ["--synthetic", "400000", "--val-ratio", "0.05"]
    SYNTHETIC = True
    print("no .txt input attached - falling back to a deterministic synthetic corpus.\n"
          "This proves the pipeline runs and the model can learn. It does not train a usable model.\n"
          "Attach a plain-text Kaggle Dataset and re-run from here for a real run.")

In [ ]:
DATA.mkdir(parents=True, exist_ok=True)
proc = run([PY, "-m", "brainlab.data.prepare", *CORPUS_ARGS, "--charlevel", "--out", DATA])
print(proc.stdout[-3000:])
if proc.returncode != 0:
    print(proc.stderr[-2000:])
    raise SystemExit(f"prepare exited {proc.returncode}")

In [ ]:
report = json.loads((DATA / "prepare_report.json").read_text())
for key in ("files", "train_files", "val_files", "val_stride", "val_fraction", "doc_sep",
            "train_documents", "val_documents", "tokens_train", "tokens_val", "train_sequences",
            "val_sequences", "seq_len", "engine", "tokenizer_vocab", "corpus_words"):
    if key in report:
        print(f"{key:18s} {report[key]}")

# prepare raises when the achieved ratio misses the target by more than 3x. File-level stride cannot
# hit 2% on a one-file corpus, so a tiny or lopsided split here means the corpus needs more files.
assert 0.005 <= report["val_fraction"] <= 0.10, f"val_fraction {report['val_fraction']} is not the configured split"
assert report["tokens_val"] > 0 and report["train_sequences"] > report["val_sequences"]
# train.py refuses to start unless val holds two full eval windows, and a window is
# eval_batches * min(micro_batch, len(val)) sequences - so the ceiling on eval_batches is val//16.
VAL_BATCHES = max(1, min(64, report["val_sequences"] // 16))
assert report["val_sequences"] >= 16, (
    f"val holds {report['val_sequences']} sequences; training needs at least 16 for one track window "
    "and one select window. Use a larger corpus, or raise --val-ratio."
)
print(f"\nsplit guard held: val_fraction on target, validation non-empty, eval_batches clamped to {VAL_BATCHES}")

### Token-span leakage

100 random val sequences, every 50-token window hashed against the train shards. A hit means validation text
appears inside training data and every number after it is inflated. `brainlab leak` exits 1 on a hit, which is
why it gates this notebook rather than being a report someone reads later.

On the synthetic fallback the check *does* fire, and that is the corpus, not the split: `synthetic_docs` reuses
a handful of template sentences, so verbatim 50-token windows recur across documents by construction. The cell
therefore reads `shared_source_files` - which is the real split-correctness signal - and treats window matches
as fatal only against a real corpus. A green leak check on synthetic text would mean the generator had stopped
producing repeatable phrases, not that your data is clean.

In [ ]:
proc = run([PY, "-m", "brainlab.data.leakage", "--train-dir", DATA / "train",
              "--val-dir", DATA / "val", "--sequences", min(100, report["val_sequences"]),
              "--span", 50, "--json"])
try:
    leak = json.loads(proc.stdout.strip().splitlines()[-1])
except (ValueError, IndexError):
    leak = None
if leak is None:
    print(proc.stdout[-1500:] or proc.stderr[-1500:])
    raise SystemExit(f"leak check exited {proc.returncode} with no report "
                     "(3 = it could not run, e.g. sequences shorter than the span)")
print(json.dumps({key: value for key, value in leak.items() if key != "examples"}, indent=2, sort_keys=True))
if leak.get("examples"):
    example = leak["examples"][0]
    # byte-level tokenizer: ids 0-255 are bytes, but the 258-vocab also carries special ids above that
    # range, which bytes() refuses - so they are counted rather than decoded.
    byte_ids = [token for token in example["tokens"] if token < 256]
    special = len(example["tokens"]) - len(byte_ids)
    text = bytes(byte_ids).decode("utf-8", "replace")
    print(f"first matched window - val sequence {example['val_sequence']}, offset {example['position']}:")
    print(f"  {text[:140]!r}" + (f"   [{special} special id(s) above the byte range]" if special else ""))

# shared_source_files is the split-correctness signal: a val file listed there means the file-level stride
# put the same file on both sides. matching_windows is a different thing - on a template-generated corpus
# every sentence recurs, so verbatim windows match while the split itself stays honest.
assert leak["shared_source_files"] == [], f"val files are inside the train shards: {leak['shared_source_files'][:5]}"
if leak["leaked"]:
    if SYNTHETIC:
        print(f"expected here: {leak['matching_windows']} windows match because synthetic_docs reuses its "
              f"template sentences verbatim. No val file is in train, so the split is honest. "
              "This check is only meaningful against a real corpus.")
    else:
        raise SystemExit("LEAK FOUND on a real corpus: validation text occurs inside the train shards, "
                         "so every metric from this split is inflated.")
else:
    print(f"clean: {leak['val_sequences_sampled']} sequences sampled, {leak['val_windows']} windows, zero matches")

## 5. Preflight: can this model learn at all?

Parity tests compare outputs to a reference. They cannot see a loss that never descends, a gradient clipped to
nothing, or an optimizer that discards every update. So: two sequences of 32 tokens, repeated for many steps.
The loss must collapse toward 0. A 33M-parameter model that cannot memorise 64 tokens is broken, and that
failure is invisible to every parity test in the repository.

On an accelerator this takes the fp16 + GradScaler path - the same one the T4 will use for real - so it also
confirms mixed precision *trains* rather than merely forward-passing.

In [ ]:
import math
import tempfile

sys.path.insert(0, str(REPO / "python"))
import numpy as np
from brainlab.data.pack import PackConfig, pack_and_write

work = Path(tempfile.mkdtemp(prefix="overfit-"))
rng = np.random.default_rng(0)
cfg = PackConfig(seq_len=32, shard_tokens=64, eos_id=1)
pack_and_write([[int(v) for v in rng.integers(2, 250, size=32)] for _ in range(2)], work / "train", "train", cfg)
pack_and_write([[int(v) for v in rng.integers(2, 250, size=32)] for _ in range(8)], work / "val", "val", cfg)

overfit = run([PY, "-m", "brainlab.train.train",
               "--train-dir", work / "train", "--val-dir", work / "val",
               "--seq-len", 32, "--micro-batch", 1, "--accum", 1,
               "--steps", 60, "--eval-every", 60, "--eval-batches", 1,
               "--save-every", 60, "--log-every", 10,
               "--device", DEVICE, "--dtype", DTYPE, "--seed", 1, "--out", work / "ckpt"])
rows = [json.loads(line) for line in overfit.stdout.splitlines() if line.startswith('{"')]
train = [row for row in rows if row.get("event") == "train"]
if not train:
    print(overfit.stdout[-2000:], overfit.stderr[-2000:])
    raise SystemExit(f"training produced no train rows (exit {overfit.returncode})")

floor = math.log(16384)
first, last = train[0], train[-1]
print(f"steps logged {len(train)}   exit {overfit.returncode}")
print(f"loss at step {first['step']:<3d} {first['loss']:.4f}")
print(f"loss at step {last['step']:<3d} {last['loss']:.4f}")
print(f"uniform floor ln(16384) = {floor:.4f}   final/floor = {last['loss'] / floor:.4f}")
print(f"grad_norm {first.get('grad_norm')} -> {last.get('grad_norm')}")
if last["loss"] >= 1.0:
    raise SystemExit(
        "FAIL: the model could not memorise 64 tokens. Something in the loss, the optimizer or the data "
        "path is wrong and no parity test would have caught it. Do not start a long run."
    )
print(f"\nok: forward, loss and optimizer move together on the {DEVICE}/{DTYPE} path")

## 6. Throughput, measured before committing to it

The configured step is `micro-batch 8 x accum 4 x seq_len 1024 = 32,768 tokens`, and the plan is 8,000 steps =
**262,144,000 tokens**. The only question this answers is whether that fits in a session.

The median of the logged `tokens_per_sec` is used, not the mean: the first steps pay for cuDNN autotuning and
drag an average down without describing any step you would actually wait for.

In [ ]:
import statistics

probe = run([PY, "-m", "brainlab.train.train",
             "--train-dir", DATA / "train", "--val-dir", DATA / "val",
             "--steps", 40, "--eval-every", 40, "--eval-batches", VAL_BATCHES,
             "--log-every", 1, "--save-every", 40, "--keep", 1,
             "--device", DEVICE, "--dtype", DTYPE, "--seed", 1,
             "--out", WORK / "ckpt-probe"])
probe_rows = [json.loads(line) for line in probe.stdout.splitlines() if line.startswith('{"')]
rate_rows = [row for row in probe_rows if row.get("event") == "train" and "tokens_per_sec" in row]
if not rate_rows:
    print(probe.stdout[-2000:], probe.stderr[-2000:])
    raise SystemExit(f"probe produced no throughput rows (exit {probe.returncode})")

rate = statistics.median(row["tokens_per_sec"] for row in rate_rows)
tokens_per_step = 8 * 4 * 1024
total_tokens = 8000 * tokens_per_step
hours = total_tokens / rate / 3600
print(f"measured {rate:,.0f} tokens/s over {len(rate_rows)} logged steps on {DEVICE}/{DTYPE}")
print(f"{tokens_per_step:,} tokens/step -> 8,000 steps = {total_tokens:,} tokens = {hours:.1f} h")
print(f"what fits in one 9 h session at this rate: {int(9 * 3600 * rate / tokens_per_step):,} steps")
for row in probe_rows:
    if row.get("event") == "eval":
        print(f"eval {row['subset']:6s} step {row['step']:3d}  val_loss {row['val_loss']:.4f}"
              f"  val_ppl {row['val_ppl']:.2f}  over {int(row['val_tokens']):,} tokens")

If the projection exceeds a session, that is not a reason to shrink the model mid-experiment. Either accept
several resumed sessions - the next cell is built for it - or run the full thing on a host without the cap.
Changing `--micro-batch` or `--accum` changes the batch the recipe was written against, so it changes what the
run *means*, not only how long it takes.

## 7. The run, with a budget and a verified resume

About interrupting, because this is where notebooks lie: **`last.pt` is written when the loop finishes, not on
a signal.** `train.py` installs no SIGTERM handler. A killed run therefore resumes from the newest
`step-NNNNNN.pt`, which is what `latest_cursor` finds - and a cell that told you to `--resume last.pt` after a
kill would hand you a checkpoint that does not exist, silently restarting the run from step 0.

Killing between saves loses up to `save_every` steps (by default one eval interval), and resume replays from
the save, not from the kill. D028 is the contract: the checkpoint carries the data cursor and the best score,
and the *config's* peak LR wins over the LR stored in the file. `just wsl-resume` proves on Linux that a
killed-and-resumed curve matches an uninterrupted one to 0.000e+00.

In [ ]:
SESSION_MINUTES = 240   # stop early and resume, instead of losing the run to the Kaggle cap
CKPT.mkdir(parents=True, exist_ok=True)
run_log = CKPT / "train.jsonl"


def latest_cursor(directory):
    found, newest = None, -1
    for candidate in sorted(list(directory.glob("step-*.pt")) + list(directory.glob("last.pt"))):
        step = int(candidate.stem.split("-")[-1]) if candidate.stem.startswith("step-") else 10**9
        if step >= newest:
            found, newest = candidate, step
    return found


cmd = [PY, "-u", "-m", "brainlab.train.train",
       "--train-dir", DATA / "train", "--val-dir", DATA / "val",
       "--steps", 8000, "--device", DEVICE, "--dtype", DTYPE,
       "--out", CKPT, "--keep", "3", "--log-every", 10,
       "--eval-every", 250, "--eval-batches", VAL_BATCHES]
prior = latest_cursor(CKPT)
if prior:
    cmd += ["--resume", prior]
    print(f"resuming from {prior.name}")
print(" ".join(str(item) for item in cmd))

with run_log.open("a", encoding="utf-8") as handle:
    proc = subprocess.Popen([str(item) for item in cmd], cwd=str(REPO),
                            stdout=handle, stderr=subprocess.STDOUT, text=True)
    try:
        returncode = proc.wait(timeout=SESSION_MINUTES * 60)
        budget_hit = False
    except subprocess.TimeoutExpired:
        print(f"budget of {SESSION_MINUTES} min reached; the child runs with -u, so every logged step is "
              f"already on disk before it stops")
        proc.terminate()
        try:
            returncode = proc.wait(timeout=120)
        except subprocess.TimeoutExpired:
            proc.kill()
            returncode = proc.wait()
        budget_hit = True

print(f"\nbudget {SESSION_MINUTES} min reached: {budget_hit}   training exit: {returncode}")
checkpoint = latest_cursor(CKPT)
if checkpoint is None:
    # save_every defaults to eval_every, so a budget shorter than one eval interval writes nothing at all.
    print("no checkpoint written yet - the first step-NNNNNN.pt lands at the save interval "
          f"(every {250} steps by default, matching --eval-every). Re-running this cell restarts from step 0.")
else:
    print(f"newest checkpoint: {checkpoint.name}")
    if budget_hit:
        print(f"budget reached; resume with  --resume {checkpoint}")
print(f"log: {run_log}")

In [ ]:
lines = run_log.read_text().splitlines() if run_log.exists() else []
rows = [json.loads(line) for line in lines if line.startswith('{"')]
train = [row for row in rows if row.get("event") == "train"]
evals = [row for row in rows if row.get("event") == "eval"]
config = next((row for row in rows if row.get("event") == "config"), {})

for key in ("git", "device", "dtype", "resumed_from", "start_step", "start_batch", "peak_lrs",
            "eval_track_window", "eval_select_window", "report_scope"):
    if key in config:
        print(f"{key:19s} {config[key]}")

if not train:
    raise SystemExit("no train rows in the log - the run never reached its first log interval")
print(f"\nsteps {train[0]['step']} -> {train[-1]['step']}   loss {train[0]['loss']:.4f} -> {train[-1]['loss']:.4f}")
for subset in ("track", "select"):
    picked = [row for row in evals if row.get("subset") == subset]
    if picked:
        print(f"{subset:6s} {len(picked)} rows  best val_loss {min(row['val_loss'] for row in picked):.4f}"
              f"  final val_ppl {picked[-1]['val_ppl']:.2f}")
track_window = config.get("eval_track_window")
select_window = config.get("eval_select_window")
print(f"\neval windows: track {track_window}, select {select_window}, over {config.get('val_sequences')} val sequences")
if track_window and select_window:
    # the two rows legitimately share a step number - they are separated by which slice of the val shards
    # they read, so disjointness is a property of the ranges, not of the steps.
    assert track_window[0] < track_window[1] <= select_window[0] < select_window[1], \
        "the select window is not disjoint from the track window"
    assert select_window[1] <= config["val_sequences"], "the select window runs past the end of the val shard"
    print(f"disjoint by data range: track owns sequences {track_window[0]}-{track_window[1]}, "
          f"select owns {select_window[0]}-{select_window[1]}, and best.pt is chosen on the second pair")

## 8. Score it, then sample from it

Export is `f32` deliberately. The tied output head is where a 16,384-way softmax gets its resolution, and the
open question in `docs/model_audit_report.md` is whether that head runs in fp32 or is cast to fp16 under
autocast. `brainlab ppl --amp` prints the dtype it actually used, which turns that from a claim into a
measurement. Change the head's precision only if the perplexity gap exceeds about 0.002 nats.

Sampling is qualitative. `--charlevel` has to match how the data was packed, or the prompt is tokenised against
a vocabulary the model never saw and the output is noise for the wrong reason.

In [ ]:
source = latest_cursor(CKPT)
best = CKPT / "best.pt"
chosen = best if best.exists() else source
if chosen is None:
    raise SystemExit("nothing to score yet: the checkpoints directory holds no step-NNNNNN.pt or best.pt. "
                     "Run the training cell past its first save interval, then come back here.")
print(f"scoring {chosen.name} ({'selection-window winner' if chosen == best else 'latest step checkpoint'})")
safetensors_out = CKPT / "brainlab-33m.safetensors"
proc = run([PY, "-m", "brainlab.export.to_safetensors", "--checkpoint", chosen,
            "--out", safetensors_out, "--dtype", "f32"], check=True)
print(proc.stdout[-1500:])

In [ ]:
plain = run([PY, "-m", "brainlab.eval.perplexity", "--model", safetensors_out,
              "--data-dir", DATA / "val", "--device", DEVICE, "--batch", 8, "--json"])
print("== fp32 (reference)")
print(plain.stdout[-1200:] or plain.stderr[-1200:])

amp = run([PY, "-m", "brainlab.eval.perplexity", "--model", safetensors_out,
           "--data-dir", DATA / "val", "--device", DEVICE, "--batch", 8, "--amp", "--json"])
print("== under autocast - this is where logits.dtype is decided")
print(amp.stdout[-1200:] or amp.stderr[-1200:])

try:
    fp32_row = json.loads([line for line in plain.stdout.splitlines() if line.startswith("{")][-1])
    amp_row = json.loads([line for line in amp.stdout.splitlines() if line.startswith("{")][-1])
except IndexError:
    print("could not parse ppl output; both runs printed above")
else:
    gap = abs(fp32_row["loss"] - amp_row["loss"])
    print(f"loss {fp32_row['loss']:.4f} (fp32) vs {amp_row['loss']:.4f} (amp), gap {gap:.5f} nats")
    print(f"logits dtype: fp32 run {fp32_row.get('logits_dtype')} / amp run {amp_row.get('logits_dtype')}")
    if fp32_row.get("logits_dtype") == amp_row.get("logits_dtype"):
        # on a CPU host perplexity falls back to fp32, so a gap of exactly zero here is the absence of a
        # test rather than a result. Reading it as "fp16 heads cost nothing" is how a measurement gets invented.
        print("NOT MEASURED: both passes returned the same logits dtype, so autocast never engaged and the fp16 "
              "head was never exercised. The zero gap above is the absence of a test, not a result. "
              "Re-run this cell on a CUDA device before drawing the 0.002-nat conclusion.")
    elif gap <= 0.002:
        print("fp16 head costs <= 0.002 nats - acceptable")
    else:
        print("gap exceeds 0.002 nats - keep the output head in fp32")

In [ ]:
for prompt, seed, temperature in (("Once upon a time", 0, 0.8), ("The little girl was", 7, 1.0)):
    proc = run([PY, "-m", "brainlab.eval.generate", "--model", safetensors_out, "--charlevel",
                "--prompt", prompt, "--max-new", 96, "--temperature", temperature,
                "--top-p", 0.9, "--top-k", 40, "--seed", seed, "--device", DEVICE])
    print(f"--- {prompt!r} (seed {seed}, temperature {temperature})")
    print(proc.stdout.strip() or proc.stderr.strip())
    print()

## What this leaves behind

`/kaggle/working` is saved as notebook output and nothing else in the container survives. Two things are worth
keeping: the checkpoints, and `train.jsonl` - its config header carries the git revision, the resolved device
and dtype, the resume cursor and the eval windows, so a number from weeks ago still traces back to the tree
that produced it.

**Do not commit any of it to git.** The repository is deliberately small; a 130 MB safetensors file multiplied
by every experiment is how a project becomes unclonable. Save the run as a Kaggle Dataset instead.

| proven here | not proven here |
|---|---|
| the container rebuilds from one script | a converged 8,000-step model, unless the run above finished |
| golden-tensor parity for every backend that could run | Rust-vs-PyTorch speed at equal threads - needs a matched interleaved benchmark |
| the fp16 + GradScaler path descends on this device | CUDA, Triton and Mojo performance, all deliberately unmeasured |
| the val split is held out and leak-clean | any comparison against a trained-BPE perplexity |
| resume restores the data cursor and best score | that `best.pt` generalises, until `ppl` runs over the whole val set |

In [ ]:
def size(path):
    return f"{path.stat().st_size / 1e6:.1f} MB" if path.exists() else "absent"


print("checkpoints")
for candidate in sorted(CKPT.glob("*.pt")) + sorted(CKPT.glob("*.safetensors")):
    print(f"  {candidate.name:32s} {size(candidate)}")
print("log   ", CKPT / "train.jsonl", size(CKPT / "train.jsonl"))
print("corpus", DATA / "prepare_report.json", size(DATA / "prepare_report.json"))
print(f"\ntotal output {sum(p.stat().st_size for p in CKPT.rglob('*') if p.is_file()) / 1e6:.0f} MB "
      f"under {CKPT} - keep it out of git")
status = subprocess.run(["git", "-C", str(REPO), "status", "--short"], capture_output=True, text=True).stdout
print("\nrepo working tree (must stay clean):")
print(status.strip() or "  clean")